<a href="https://colab.research.google.com/github/PattaraphonD/Data-Science-Projects/blob/main/RAG_for_summarizing_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages

In [ ]:
!pip install openai==0.28

In [ ]:
!pip install pymupdf

In [ ]:
!pip install PyPDF2


In [ ]:
!pip install faiss-cpu

## Import Libraries

In [ ]:
import os
import requests
import openai
import pickle
import numpy as np
import faiss
import PyPDF2
import pandas as pd
from google.colab import drive, userdata
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass

## Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Connecting API

### For API key security

In [ ]:
openai_api_key = getpass('Enter your OpenAI API key: ')

Enter your OpenAI API key: ··········


In [ ]:
openai.api_key = openai_api_key

## Create RAG
  - Input PDF
  - Prompting to Analyse PDF

In [ ]:
# ฟังก์ชันสำหรับอ่านข้อมูลจากไฟล์ PDF ใน Google Drive
def read_pdf_from_drive(pdf_path):
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            full_text = ''
            for page in reader.pages:
                full_text += page.extract_text() + "\n"

        if not full_text.strip():
            print("No text content found in the PDF file.")
            return "No content to analyze."

        return full_text
    except Exception as e:
        print(f"Error during PDF reading: {e}")
        return "Error during PDF reading"

# ฟังก์ชันสำหรับแบ่งข้อความออกเป็นชิ้นเล็ก ๆ (Chunk)
def split_text_into_chunks(text, chunk_size=1000, overlap=200):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# ฟังก์ชันสำหรับสร้าง embeddings
def get_embeddings(text):
    try:
        response = openai.Embedding.create(
            model="text-embedding-ada-002",
            input=text
        )
        embeddings = response['data'][0]['embedding']
        return embeddings
    except Exception as e:
        print(f"Error generating embeddings: {e}")
        return None

# ฟังก์ชันสำหรับสร้างและจัดเก็บ embeddings ลงในฐานข้อมูล vector
def build_vector_store(chunks):
    embeddings_list = []
    for chunk in chunks:
        embedding = get_embeddings(chunk)
        if embedding is not None:
            embeddings_list.append(embedding)

    d = len(embeddings_list[0])  # Dimensionality of embeddings
    index = faiss.IndexFlatL2(d)
    embeddings_array = np.array(embeddings_list).astype('float32')
    index.add(embeddings_array)

    return index, chunks

# ฟังก์ชันหลัก
def main():
    print("=== PDF Reader with Vector Store Creation ===")

    # เชื่อมต่อ Google Drive
    drive.mount('/content/drive')

    # รับเส้นทาง PDF จากผู้ใช้
    pdf_path = input("Enter the path to the PDF file on your Google Drive (e.g., /content/drive/MyDrive/sample.pdf): ").strip()

    print("\nReading and processing PDF content...")

    # อ่านเนื้อหา PDF
    full_content = read_pdf_from_drive(pdf_path)

    if "Error" in full_content:
        print("Error occurred during PDF reading.")
        return

    # แบ่งเนื้อหาออกเป็นชิ้น (chunks)
    chunks = split_text_into_chunks(full_content)

    print("Building vector store for the chunks...")
    index, stored_chunks = build_vector_store(chunks)

    # บันทึก Vector Store และ Chunks ลงในไฟล์
    with open('/content/vector_store.pkl', 'wb') as f:
        pickle.dump(index, f)

    with open('/content/chunks.pkl', 'wb') as f:
        pickle.dump(stored_chunks, f)

    print("\nVector store and chunks saved successfully!")

if __name__ == "__main__":
    main()


=== PDF Reader with Vector Store Creation ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter the path to the PDF file on your Google Drive (e.g., /content/drive/MyDrive/sample.pdf): /content/drive/MyDrive/RAG/swu_calendar_67.pdf

Reading and processing PDF content...
Building vector store for the chunks...
Error generating embeddings: This model's maximum context length is 8192 tokens, however you requested 8804 tokens (8804 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.

Vector store and chunks saved successfully!


In [ ]:
# ฟังก์ชันสำหรับสร้าง embeddings ของ prompt
def get_embeddings(text):
    try:
        response = openai.Embedding.create(
            model="text-embedding-ada-002",
            input=text
        )
        embeddings = response['data'][0]['embedding']
        return embeddings
    except Exception as e:
        print(f"Error generating embeddings: {e}")
        return None

# ฟังก์ชันสำหรับเรียกคืนชิ้นที่เกี่ยวข้องกับ query ของผู้ใช้
def retrieve_relevant_chunks(user_prompt, index, chunks, top_k=5):
    query_embedding = get_embeddings(user_prompt)
    if query_embedding is None:
        return []
    distances, indices = index.search(np.array([query_embedding]).astype('float32'), top_k)
    relevant_chunks = [chunks[i] for i in indices[0]]
    return relevant_chunks

# ฟังก์ชันสำหรับจำแนกเนื้อหาตาม prompt ที่กำหนดโดยผู้ใช้
def classify_content(text, user_prompt):
    prompt = f"""
    {user_prompt}
    Content: "{text}"
    Provide a response or classification based on the user's prompt.
    """

    try:
        response = openai.ChatCompletion.create(
            model="chatgpt-4o-latest",
            messages=[{"role": "user", "content": prompt}]
        )
        classification = response['choices'][0]['message']['content'].strip()
        return classification
    except Exception as e:
        print(f"Error with OpenAI API call: {e}")
        return "Error"

# ฟังก์ชันหลัก
def main():
    print("=== RAG Query System ===")

    # โหลด Vector Store และ Chunks ที่จัดเก็บไว้
    with open('/content/vector_store.pkl', 'rb') as f:
        index = pickle.load(f)

    with open('/content/chunks.pkl', 'rb') as f:
        stored_chunks = pickle.load(f)

    while True:
        user_prompt = input("\nEnter your prompt for AI analysis (or type 'exit' to quit): ").strip()

        if user_prompt.lower() in ['exit', 'quit']:
            print("\nExiting the program. Goodbye!")
            break

        print("Retrieving relevant chunks for the user's query...")
        relevant_chunks = retrieve_relevant_chunks(user_prompt, index, stored_chunks)

        if not relevant_chunks:
            print("No relevant chunks found.")
            continue

        combined_relevant_text = ' '.join(relevant_chunks)

        print("Classifying content with AI...")
        analysis_result = classify_content(combined_relevant_text, user_prompt)

        print("\n=== Results ===")
        print(f"\nAI Analysis Result:\n{analysis_result}")

if __name__ == "__main__":
    main()

=== RAG Query System ===

Enter your prompt for AI analysis (or type 'exit' to quit): สรุปวันสำคัญของมหาวิทยาลัยในปีการศึกษานี้ โดยจัดกลุ่มตามหัวข้อ เช่น วันเปิดภาคเรียน วันสอบ และวันหยุดนักขัตฤกษ์
Retrieving relevant chunks for the user's query...
Classifying content with AI...

=== Results ===

AI Analysis Result:
**สรุปวันสำคัญของมหาวิทยาลัยในปีการศึกษา 2567 โดยจัดกลุ่มตามหัวข้อ**

---

### 1. **วันเปิดภาคเรียน**  
- **ภาคเรียนที่ 1**: วันที่ **19 สิงหาคม 2567**  
- **ภาคเรียนที่ 2**: วันที่ **3 กุมภาพันธ์ 2568**

---

### 2. **วันสอบ**
#### สอบคุณสมบัติ/ประมวลความรู้  
- **ภาคเรียนที่ 1**: วันสุดท้าย **10 มกราคม 2568**  
- **ภาคเรียนที่ 2**: วันสุดท้าย **13 มิถุนายน 2568**

#### สอบเค้าโครงปริญญานิพนธ์/สารนิพนธ์  
- **ภาคเรียนที่ 1**  
  - วันสุดท้ายยื่นแต่งตั้งกรรมการสอบ: **13 ธันวาคม 2567**  
  - วันสุดท้ายสอบ: **27 ธันวาคม 2567**  
- **ภาคเรียนที่ 2**  
  - วันสุดท้ายยื่นแต่งตั้งกรรมการสอบ: **13 มิถุนายน 2568**  
  - วันสุดท้ายสอบ: **30 มิถุนายน 2568**

#### สอบปากเปล่าปริญญานิพ